# Model Training

Train ML models for price prediction.

In [ ]:
import pandas as pd
import sys
sys.path.append('../src')

from src.modeling.train_model import prepare_features, train_random_forest, save_model
from src.modeling.evaluate_model import evaluate_classification, plot_confusion_matrix, plot_feature_importance

In [2]:
%pip install seaborn --quiet

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Load features
ticker = 'AAPL'
from pathlib import Path
# Try multiple relative locations (notebook may live in a subfolder)
candidates = [
    Path('../data/indicators') / f'{ticker}_features.csv',
    Path('../../data/indicators') / f'{ticker}_features.csv',
    Path('../../../data/indicators') / f'{ticker}_features.csv',
    Path('../../../../data/indicators') / f'{ticker}_features.csv',
    Path('../../../../data') / 'indicators' / f'{ticker}_features.csv',
]
for p in candidates:
    if p.exists():
        data = pd.read_csv(p, index_col=0, parse_dates=True)
        print(f'Loaded features from: {p}')
        break
else:
    raise FileNotFoundError(
        f'Could not find {ticker}_features.csv. Checked paths: {', '.join(str(x) for x in candidates)}'
    )

# Prepare data
X_train, X_test, y_train, y_test, scaler, features = prepare_features(data)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')
print(f'Features: {len(features)}')

Loaded features from: ..\..\..\..\data\indicators\AAPL_features.csv
2025-12-14 19:31:35 - modeling.train_model - INFO - Training set: (1109, 44), Test set: (278, 44)


Training samples: 1109
Test samples: 278
Features: 44


c:\Users\Nihar\Documents\GitHub\oop\SnowMore\algo-trading-project\notebooks\../src\modeling\train_model.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['target'] = (df['Close'].shift(-1) > df['Close']).astype(int)


In [6]:
# Train model
model = train_random_forest(X_train, y_train, n_estimators=100)

2025-12-14 19:33:45 - modeling.train_model - INFO - Training Random Forest model...
2025-12-14 19:33:46 - modeling.train_model - INFO - CV Score: 0.4752 (+/- 0.0235)


In [7]:
# Evaluate
y_pred = model.predict(X_test)
metrics = evaluate_classification(y_test, y_pred, model_name='Random Forest')

2025-12-14 19:33:51 - modeling.evaluate_model - INFO - 
Random Forest Performance:
2025-12-14 19:33:51 - modeling.evaluate_model - INFO - Accuracy: 0.9496
2025-12-14 19:33:51 - modeling.evaluate_model - INFO - Precision: 0.6667
2025-12-14 19:33:51 - modeling.evaluate_model - INFO - Recall: 0.2500
2025-12-14 19:33:51 - modeling.evaluate_model - INFO - F1 Score: 0.3636
2025-12-14 19:33:51 - modeling.evaluate_model - INFO - 
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.99      0.97       262
           1       0.67      0.25      0.36        16

    accuracy                           0.95       278
   macro avg       0.81      0.62      0.67       278
weighted avg       0.94      0.95      0.94       278



In [8]:
# Save model
save_model(model, scaler, features, ticker, output_dir='../models')
print('Model saved successfully!')

2025-12-14 19:35:34 - modeling.train_model - INFO - Model saved to ..\models\AAPL_model.pkl
Model saved successfully!
